# **Amazon Fine Food Reviews - Exploratory Data Analysis**

### Project : Fake Review Detection & Sentiment Analysis
### Dataset : Amazon Fine Food Reviews (kaggle)
### Goal : Understand pattern in reviews to build a fake review detector

# 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import re
import string
from collections import Counter
from wordcloud import WordCloud

# Text
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download NLTK data
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download("wordnet")

# Color scheme
COLORS = {
    'primary'  : '#2196F3',   # main blue
    'light'    : '#90CAF9',   # light blue
    'dark'     : '#1565C0',   # dark blue
    'muted'    : '#B0BEC5',   # grey
    'white'    : '#FFFFFF',
}

RATING_COLORS = [
    '#90CAF9',   # 1 star - light blue
    '#64B5F6',   # 2 star
    '#2196F3',   # 3 star - medium blue
    '#1976D2',   # 4 star
    '#1565C0',   # 5 star - dark blue
]

SENT_COLORS = {
    'Positive' : '#1976D2',   # blue
    'Neutral'  : '#90CAF9',   # light blue
    'Negative' : '#B0BEC5',   # grey
}

# ── Clean Style ──────────────────────
plt.rcParams.update({
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
    'axes.edgecolor'    : '#E0E0E0',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.spines.left'  : False,
    'axes.labelsize'    : 11,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.titlecolor'   : '#212121',
    'xtick.labelsize'   : 10,
    'ytick.labelsize'   : 10,
    'xtick.color'       : '#757575',
    'ytick.color'       : '#757575',
    'grid.alpha'        : 0.3,
    'grid.linestyle'    : '--',
    'grid.color'        : '#E0E0E0',
    'font.family'       : 'sans-serif',
})

# Create output directories
os.makedirs("../plots", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save plot function - ALL PLOTS GO TO ONE FOLDER
def save_plots(filename):
    """Save plots to ../plots folder"""
    plt.savefig(f"../plots/{filename}.png", dpi=150, bbox_inches="tight")
    plt.show()

# Apply base style to axes
def apply_base_style(axes):
    """Apply consistent styling to plot axes"""
    if isinstance(axes, np.ndarray):
        for ax in axes.flat:
            ax.set_facecolor("#F8F9FA")
            ax.set_facecolor("white")
            for spine in ax.spines.values():
                spine.set_edgecolor("#CCCCCC")
            ax.grid(axis="y", color="#CCCCCC", alpha=0.5, linestyle="--")
            ax.tick_params(colors="#444444")
    else:
        axes.set_facecolor("#F8F9FA")
        axes.set_facecolor("white")
        for spine in axes.spines.values():
            spine.set_edgecolor("#CCCCCC")
        axes.grid(axis="y", color="#CCCCCC", alpha=0.5, linestyle="--")
        axes.tick_params(colors="#444444")

print('✅ All libraries imported successfully!')

# 2. Load Dataset

In [ ]:
# Try loading from local path first
try:
    df = pd.read_csv("../data/Reviews.csv")
    print("✅ Dataset loaded from local path")
except FileNotFoundError:
    try:
        import kagglehub
        path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
        df = pd.read_csv(f"{path}/Reviews.csv")
        print("✅ Dataset downloaded from Kaggle")
    except Exception as e:
        print(f"❌ Error loading dataset: {e}")
        print("Please ensure the Reviews.csv file exists in ../data/ folder")

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
print("Column names:")
print(df.columns.tolist())

# 3. Basic Info & Data Quality

In [ ]:
print("="*50)
print("Dataset Info")
print("="*50)
df.info()

print("\n" + "="*50)
print("Missing Values")
print("="*50)

missing = df.isnull().sum()
missing_percent = missing / len(df) * 100
missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing %": missing_percent
})
print(missing_df[missing_df['missing_count'] > 0])

print("\n" + "="*50)
print("Duplicate Values")
print("="*50)
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate reviews (same user_id + product_id): {df.duplicated(subset=['ProductId', 'UserId']).sum()}")

# 4. Data Cleaning

In [ ]:
# Drop duplicates and handle missing values
df.drop_duplicates(inplace=True)
df.dropna(subset=["Summary", "Text"], inplace=True)

# Convert time to datetime
df["ReviewDate"] = pd.to_datetime(df["Time"], unit="s")
df["Year"] = df["ReviewDate"].dt.year
df["Month"] = df["ReviewDate"].dt.month
df["DayOfWeek"] = df["ReviewDate"].dt.day_name()

# Review Length Features
df["Text_Length"] = df["Text"].apply(len)
df["WordCount"] = df["Text"].apply(lambda x: len(str(x).split()))
df["SummaryLength"] = df["Summary"].apply(len)

# Helpfulness Ratio
df["Helpfulness_Ratio"] = df.apply(
    lambda x: x["HelpfulnessNumerator"] / x["HelpfulnessDenominator"]
    if x["HelpfulnessDenominator"] > 0 else 0, axis=1)

print(f'✅ Clean dataset shape: {df.shape}')

# Sentiment classification
df["Sentiment"] = df["Score"].apply(lambda x: "Positive" if x >= 4
                                     else ("Negative" if x <= 2 else "Neutral"))

print("\nSentiment Distribution:")
print(df["Sentiment"].value_counts())

# Summary statistics
print("\nBasic Statistics:")
print(df[['Score', 'Text_Length', 'WordCount', 'Helpfulness_Ratio', 'Year']].describe().round(2))

# 5. ⭐ Ratings Distribution Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
apply_base_style(ax)

# Count plot
score_counts = df["Score"].value_counts().sort_index()
colors = RATING_COLORS

bars = ax.bar(score_counts.index, score_counts.values, color=colors,
              edgecolor="white", linewidth=1.5)

ax.set_title("Rating Distribution", fontsize=14, fontweight='bold')
ax.set_xlabel("Star Rating", fontsize=12)
ax.set_ylabel("Number of Reviews", fontsize=12)

# Add value labels on bars
for i, (idx, val) in enumerate(score_counts.items()):
    ax.text(idx, val + 5000, f"{val:,}", ha="center", fontsize=10, fontweight='bold')

plt.tight_layout()
save_plots("01_rating_distribution")

# 6. 📊 Reviews Over Years

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
apply_base_style(axes)

# Reviews per year
yearly = df.groupby('Year').size()
axes[0].fill_between(yearly.index, yearly.values, alpha=0.4, color='#1565C0')
axes[0].plot(yearly.index, yearly.values, 'o-', color="#1565C0", linewidth=2, markersize=8)
axes[0].set_title('Total Reviews Per Year', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Year', fontsize=11)
axes[0].set_ylabel('Number of Reviews', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Add value labels
for year, count in yearly.items():
    axes[0].text(year, count + 2000, f"{count:,}", ha="center", fontsize=9)

# Average rating per year
avg_rating_year = df.groupby("Year")["Score"].mean()
axes[1].plot(avg_rating_year.index, avg_rating_year.values, 's-', 
             color='#9b59b6', linewidth=2.5, markersize=8, label='Avg Rating')
axes[1].axhline(y=avg_rating_year.mean(), color='red', linestyle='--', 
                alpha=0.7, linewidth=2, label=f'Overall Avg: {avg_rating_year.mean():.2f}')
axes[1].set_title('Average Star Rating Per Year', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Year', fontsize=11)
axes[1].set_ylabel('Average Rating', fontsize=11)
axes[1].set_ylim(3.5, 4.8)
axes[1].legend(fontsize=10, loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
save_plots("02_reviews_per_year")

# 7. 📊 Text Length Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
apply_base_style(axes)

# Word count by rating
df.boxplot(column='WordCount', by='Score', ax=axes[0],
           boxprops=dict(color='#2c3e50'), medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Word Count by Star Rating', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Star Rating', fontsize=11)
axes[0].set_ylabel('Word Count', fontsize=11)
axes[0].set_ylim(0, 300)
axes[0].get_figure().suptitle('')  # Remove the automatic title

# Word count distribution
axes[1].hist(df['WordCount'].clip(upper=500), bins=50, color='#9b59b6', 
             edgecolor='white', alpha=0.8)
axes[1].axvline(df['WordCount'].median(), color='red', linestyle='--', 
                linewidth=2, label=f'Median: {df["WordCount"].median():.0f}')
axes[1].axvline(df['WordCount'].mean(), color='orange', linestyle='--', 
                linewidth=2, label=f'Mean: {df["WordCount"].mean():.0f}')
axes[1].set_title('Distribution of Review Word Count', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count (clipped at 500)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].legend(fontsize=10)

plt.tight_layout()
save_plots("03_text_length_analysis")

# Insights
print("\n📊 Average word count by rating:")
avg_by_score = df.groupby('Score')['WordCount'].mean()
for score, avg in avg_by_score.items():
    print(f'  {score}★ → {avg:.1f} words')

# 8. 📊 User Behavior Analysis

In [ ]:
user_review_counts = df.groupby("UserId").size().reset_index(name="ReviewCount")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
apply_base_style(axes)

# Distribution of reviews per user
axes[0].hist(user_review_counts['ReviewCount'].clip(upper=50), bins=40,
             color='#1abc9c', edgecolor='white', alpha=0.85)
axes[0].set_title('Reviews Per User (clipped @50)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Reviews', fontsize=11)
axes[0].set_ylabel('Number of Users', fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Top 20 prolific reviewers
top_users = user_review_counts.nlargest(20, 'ReviewCount')
axes[1].barh(range(20), top_users['ReviewCount'].values, color='#e67e22', alpha=0.85)
axes[1].set_yticks(range(20))
axes[1].set_yticklabels([uid[:12]+'...' for uid in top_users['UserId']], fontsize=8)
axes[1].set_title('Top 20 Most Active Reviewers', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Reviews', fontsize=11)
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')

# Add value labels
for i, v in enumerate(top_users['ReviewCount'].values):
    axes[1].text(v + 2, i, str(v), va='center', fontsize=8)

plt.tight_layout()
save_plots("04_user_behavior_analysis")

print(f"\n👥 User Statistics:")
print(f"  Total unique users: {df['UserId'].nunique():,}")
print(f"  Avg reviews per user: {df.groupby('UserId').size().mean():.2f}")
print(f"  Max reviews by one user: {user_review_counts['ReviewCount'].max()}")

# 9. 🎯 Text Cleaning (NLP Preprocessing)

In [ ]:
def clean_text(text):
    """Clean and preprocess text for NLP analysis"""
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Remove URLs
    text = re.sub(r'http\S+', ' ', text)
    
    # Remove special characters and keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize and lemmatize
    lemmatizer = WordNetLemmatizer()
    stopwords_set = set(stopwords.words("english"))
    stopwords_set.update(["br", "one", "get", "also", "would", "product", "like", "food", "taste"])
    
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stopwords_set and len(t) > 2]
    
    return " ".join(tokens)

print("🔄 Cleaning text data...")
df["CleanText"] = df["Text"].apply(clean_text)
print("✅ Text cleaning completed!")

print("\n📝 Original Text Sample:")
print(df["Text"].iloc[0][:200])

print("\n✨ Cleaned Text Sample:")
print(df["CleanText"].iloc[0])

In [ ]:
print("\n🔍 First 5 cleaned texts:")
for i, text in enumerate(df["CleanText"].head(), 1):
    print(f"{i}. {text[:100]}...")

# 10. 📊 Word Clouds - Positive, Negative & Neutral Reviews

In [ ]:
# Create wordclouds for each sentiment
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sentiments = ['Positive', 'Neutral', 'Negative']
colors_list = ['#1976D2', '#90CAF9', '#B0BEC5']

for idx, (sentiment, color) in enumerate(zip(sentiments, colors_list)):
    # Get text for this sentiment
    sentiment_text = " ".join(df[df['Sentiment'] == sentiment]['CleanText'].values)
    
    # Create wordcloud
    wordcloud = WordCloud(width=400, height=300, 
                          background_color='white',
                          colormap='Blues',
                          max_words=100).generate(sentiment_text)
    
    # Display
    axes[idx].imshow(wordcloud, interpolation='bilinear')
    axes[idx].set_title(f'{sentiment} Reviews', fontsize=13, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
save_plots("05_wordclouds_by_sentiment")

print(f"\n📊 Sentiment Distribution:")
for sentiment in sentiments:
    count = len(df[df['Sentiment'] == sentiment])
    pct = (count / len(df)) * 100
    print(f"  {sentiment}: {count:,} ({pct:.1f}%)")

# 11. 📊 Sentiment Distribution by Rating

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
apply_base_style(axes)

# Sentiment counts
sentiment_counts = df['Sentiment'].value_counts()
axes[0].pie(sentiment_counts.values, labels=sentiment_counts.index, 
            autopct='%1.1f%%', colors=[SENT_COLORS[s] for s in sentiment_counts.index],
            startangle=90, textprops={'fontsize': 11, 'weight': 'bold'})
axes[0].set_title('Overall Sentiment Distribution', fontsize=13, fontweight='bold')

# Sentiment by score (stacked bar)
sentiment_by_score = pd.crosstab(df['Score'], df['Sentiment'])
sentiment_by_score.plot(kind='bar', stacked=True, ax=axes[1],
                        color=[SENT_COLORS[s] for s in sentiment_by_score.columns],
                        edgecolor='white', linewidth=1.5)
axes[1].set_title('Sentiment by Star Rating', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Star Rating', fontsize=11)
axes[1].set_ylabel('Number of Reviews', fontsize=11)
axes[1].legend(title='Sentiment', fontsize=10)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
save_plots("06_sentiment_distribution")

# 12. 📊 Helpfulness Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
apply_base_style(axes)

# Helpfulness by score
helpfulness_by_score = df.groupby('Score')['Helpfulness_Ratio'].mean()
axes[0].bar(helpfulness_by_score.index, helpfulness_by_score.values, 
            color=RATING_COLORS, edgecolor='white', linewidth=1.5)
axes[0].set_title('Average Helpfulness Ratio by Rating', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Star Rating', fontsize=11)
axes[0].set_ylabel('Helpfulness Ratio', fontsize=11)
axes[0].set_ylim(0, 1)

# Add value labels
for i, v in enumerate(helpfulness_by_score.values):
    axes[0].text(helpfulness_by_score.index[i], v + 0.02, f'{v:.3f}', 
                ha='center', fontsize=9)

# Helpfulness by sentiment
helpfulness_by_sentiment = df.groupby('Sentiment')['Helpfulness_Ratio'].mean().sort_values(ascending=False)
colors_sentiment = [SENT_COLORS[s] for s in helpfulness_by_sentiment.index]
axes[1].barh(helpfulness_by_sentiment.index, helpfulness_by_sentiment.values,
             color=colors_sentiment, edgecolor='white', linewidth=1.5)
axes[1].set_title('Average Helpfulness Ratio by Sentiment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Helpfulness Ratio', fontsize=11)
axes[1].set_xlim(0, 1)

# Add value labels
for i, v in enumerate(helpfulness_by_sentiment.values):
    axes[1].text(v + 0.02, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
save_plots("07_helpfulness_analysis")

# 13. 📊 Temporal Analysis - Reviews by Month

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
apply_base_style(axes)

# Create date column
df['YearMonth'] = df['ReviewDate'].dt.to_period('M')

# Reviews by year and month
monthly_reviews = df.groupby('YearMonth').size()
axes[0].plot(range(len(monthly_reviews)), monthly_reviews.values, 
             color='#1565C0', linewidth=2, marker='o', markersize=3, alpha=0.7)
axes[0].fill_between(range(len(monthly_reviews)), monthly_reviews.values, 
                     alpha=0.3, color='#1565C0')
axes[0].set_title('Reviews Over Time (Monthly)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Reviews', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Reviews by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
reviews_by_day = df['DayOfWeek'].value_counts().reindex(day_order)
axes[1].bar(range(7), reviews_by_day.values, 
           color='#9b59b6', edgecolor='white', linewidth=1.5, alpha=0.85)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_order, rotation=45)
axes[1].set_title('Reviews by Day of Week', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Reviews', fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels
for i, v in enumerate(reviews_by_day.values):
    axes[1].text(i, v + 500, f"{v:,}", ha='center', fontsize=9)

plt.tight_layout()
save_plots("08_temporal_analysis")

# 14. 📊 Summary Statistics & Insights

In [ ]:
print("\n" + "="*60)
print("🎯 AMAZON REVIEWS ANALYSIS - SUMMARY INSIGHTS")
print("="*60)

print(f"\n📊 Dataset Overview:")
print(f"  Total reviews: {len(df):,}")
print(f"  Unique users: {df['UserId'].nunique():,}")
print(f"  Unique products: {df['ProductId'].nunique():,}")
print(f"  Time span: {df['Year'].min()} - {df['Year'].max()}")

print(f"\n⭐ Rating Statistics:")
print(f"  Average rating: {df['Score'].mean():.2f} / 5")
print(f"  Median rating: {df['Score'].median():.0f}")
print(f"  Mode rating: {df['Score'].mode()[0]:.0f}")
print(f"  Std deviation: {df['Score'].std():.2f}")

print(f"\n😊 Sentiment Breakdown:")
for sentiment in ['Positive', 'Negative', 'Neutral']:
    count = len(df[df['Sentiment'] == sentiment])
    pct = (count / len(df)) * 100
    print(f"  {sentiment}: {count:,} ({pct:.1f}%)")

print(f"\n📝 Review Text Statistics:")
print(f"  Avg word count: {df['WordCount'].mean():.1f} words")
print(f"  Avg character count: {df['Text_Length'].mean():.0f} characters")
print(f"  Min word count: {df['WordCount'].min()} words")
print(f"  Max word count: {df['WordCount'].max()} words")

print(f"\n👍 Helpfulness Metrics:")
print(f"  Avg helpfulness ratio: {df['Helpfulness_Ratio'].mean():.3f}")
print(f"  Reviews with helpfulness data: {(df['HelpfulnessDenominator'] > 0).sum():,}")

print(f"\n📈 Top Insights:")
print(f"  • 5-star reviews are most common ({(df['Score'] == 5).sum():,} reviews)")
print(f"  • Average review length increases with lower ratings")
print(f"  • {(df['Score'] >= 4).sum():,} reviews are positive ({((df['Score'] >= 4).sum() / len(df) * 100):.1f}%)")
print(f"  • Peak review activity: {df['Year'].mode()[0]} with {df[df['Year'] == df['Year'].mode()[0]].shape[0]:,} reviews")

print("\n" + "="*60)

# 15. 💾 Save Cleaned Dataset

In [ ]:
# Save the cleaned dataset
output_path = "../data/cleaned_reviews.csv"
df.to_csv(output_path, index=False)
print(f"✅ Cleaned dataset saved to: {output_path}")
print(f"   File size: {os.path.getsize(output_path) / (1024**2):.2f} MB")

# Save summary statistics
summary_stats = {
    'Total Reviews': len(df),
    'Unique Users': df['UserId'].nunique(),
    'Unique Products': df['ProductId'].nunique(),
    'Average Rating': df['Score'].mean(),
    'Average Word Count': df['WordCount'].mean(),
    'Positive Reviews': len(df[df['Sentiment'] == 'Positive']),
    'Negative Reviews': len(df[df['Sentiment'] == 'Negative']),
    'Neutral Reviews': len(df[df['Sentiment'] == 'Neutral']),
}

summary_df = pd.DataFrame(list(summary_stats.items()), columns=['Metric', 'Value'])
summary_df.to_csv('../data/analysis_summary.csv', index=False)
print(f"✅ Summary statistics saved to: ../data/analysis_summary.csv")

print(f"\n📁 All plots saved to: ../plots/")

# 16. 📋 Next Steps

Now that we have completed the EDA:

1. **Feature Engineering**: Create TF-IDF vectors from cleaned text
2. **Model Training**: Build sentiment classification models
3. **Fake Review Detection**: Identify suspicious review patterns
4. **Model Evaluation**: Test accuracy, precision, recall
5. **Deployment**: Build API for real-time sentiment prediction

✅ All visualizations have been saved to the `../plots/` folder!